# 5G Network Intelligence - standalone cloud setup
Source is embedded: no GitHub token, source ZIP or dataset upload is required.
Colab: select a GPU runtime. Kaggle: enable Internet and select a GPU accelerator.
Run all cells. Drive authorization must be completed by the account owner.
Source base commit: `94d4a5de60b1024492552612fbc4dc46677d5305`; bundle SHA256: `da25c0d33bbbd59817a0a45a381a2acc737136bba8db00a201c9d6ed1bc5ea5d`.
This checks infrastructure only; it does not train a model.


In [ ]:
from pathlib import Path
import base64
import hashlib
import io
import os
import subprocess
import sys
import zipfile

IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or Path('/kaggle/working').is_dir()
IS_COLAB = not IS_KAGGLE and 'COLAB_RELEASE_TAG' in os.environ
if not (IS_COLAB or IS_KAGGLE):
    raise RuntimeError('Open this notebook in Colab or Kaggle')
WORK = Path('/kaggle/working' if IS_KAGGLE else '/content')
PROJECT = WORK / '5g-network-intelligence'
os.environ['SOURCE_GIT_COMMIT'] = '94d4a5de60b1024492552612fbc4dc46677d5305'
os.environ['SOURCE_BUNDLE_SHA256'] = 'da25c0d33bbbd59817a0a45a381a2acc737136bba8db00a201c9d6ed1bc5ea5d'
source = base64.b64decode('...')
assert hashlib.sha256(source).hexdigest() == 'da25c0d33bbbd59817a0a45a381a2acc737136bba8db00a201c9d6ed1bc5ea5d', 'Source bundle hash mismatch'
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(source)) as archive:
    for member in archive.infolist():
        if not (PROJECT / member.filename).resolve().is_relative_to(PROJECT.resolve()):
            raise ValueError('Unsafe archive member')
    archive.extractall(PROJECT)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev,ml,neural]'], check=True)
print('Project ready:', PROJECT)


## Persistent checkpoints
Colab will ask for Drive authorization. On Kaggle, save a notebook version with
outputs before ending the session; /kaggle/working is not persistent across fresh sessions.


In [ ]:
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/5g-network-intelligence/checkpoints')
elif IS_KAGGLE:
    CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
else:
    raise RuntimeError('Use a Colab or Kaggle notebook runtime')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
probe = CHECKPOINT_DIR / '.write-probe'
probe.write_text('checkpoint storage writable')
assert probe.read_text() == 'checkpoint storage writable'
probe.unlink()
print('Checkpoint directory:', CHECKPOINT_DIR)


In [ ]:
from scripts.check_runtime import inspect_runtime
import json
runtime = inspect_runtime()
runtime['checkpoint_dir'] = str(CHECKPOINT_DIR)
(CHECKPOINT_DIR / 'runtime.json').write_text(json.dumps(runtime, indent=2))
print(json.dumps(runtime, indent=2))
if not runtime['cuda_available']:
    raise RuntimeError('GPU not available to PyTorch. Select a GPU runtime before Week 4 training.')


## Week 1 pipeline (CPU)
Optional: this downloads the public pinned dataset to the notebook runtime.
GPU epoch time, quota and training throughput remain unverified until the Week 4 pilot.


In [ ]:
subprocess.run([sys.executable, '-m', 'scripts.prepare_data', '--step', 'all'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
